# Modelo RNN con metodo por Transecto y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 9: Modelo RNN (encoder-decoder GRU) para predicción horaria de O3 a 72h.
Entrada: datasets 3D normalizados (DL) generados en código 5/6.
Salida: modelos entrenados, métricas, gráficas y archivos de resultados.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, RepeatVector, TimeDistributed, Reshape, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import keras_tuner as kt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
DATA_DIR = os.path.join(BASE_DIR, "windows_partitioned")
OUTPUT_DIR = os.path.join(BASE_DIR, "models", "rnn")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parámetros
WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)

# Hiperparámetros a explorar (se puede ajustar)
HP_UNITS = [32, 64, 128]
HP_DROPOUT = [0.0, 0.2, 0.3]
HP_LR = [1e-3, 5e-4, 1e-4]
HP_BATCH_SIZE = [32, 64]
HP_EPOCHS = 100  # máximo, con early stopping

# Métricas (en unidades originales)
def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan

def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def build_model(hp, input_shape, output_dim=72):
    """
    Construye modelo encoder-decoder GRU con hiperparámetros.
    hp: objeto de keras_tuner o diccionario con valores.
    """
    if isinstance(hp, dict):
        # Modo fijo (para después de búsqueda)
        units = hp['units']
        dropout = hp['dropout']
        lr = hp['lr']
    else:
        # Modo búsqueda
        units = hp.Choice('units', HP_UNITS)
        dropout = hp.Choice('dropout', HP_DROPOUT)
        lr = hp.Choice('lr', HP_LR)

    # Encoder
    encoder_inputs = Input(shape=input_shape, name='encoder_input')
    encoder = GRU(units, return_state=True, dropout=dropout,
                  recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    encoder_outputs, state_h = encoder(encoder_inputs)

    # Decoder: repetir estado para 72 pasos
    decoder_repeated = RepeatVector(output_dim)(state_h)  # (batch, 72, units)
    decoder_gru = GRU(units, return_sequences=True, dropout=dropout,
                      recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    decoder_outputs = decoder_gru(decoder_repeated, initial_state=state_h)

    # Capa densa por paso
    decoder_dense = TimeDistributed(Dense(1))(decoder_outputs)  # (batch, 72, 1)
    outputs = Reshape((output_dim,))(decoder_dense)  # (batch, 72)

    model = Model(encoder_inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

def train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                           scaler_y, station_name, output_subdir, is_global=False):
    """
    Búsqueda de hiperparámetros, entrenamiento del mejor modelo y evaluación.
    """
    print(f"\n--- Entrenando RNN para {station_name} ---")
    input_shape = (X_train.shape[1], X_train.shape[2])  # (72, n_features)

    # 1. Búsqueda de hiperparámetros con keras_tuner (RandomSearch)
    tuner = kt.RandomSearch(
        hypermodel=lambda hp: build_model(hp, input_shape),
        objective='val_loss',
        max_trials=len(HP_UNITS)*len(HP_DROPOUT)*len(HP_LR),  # todas combinaciones
        executions_per_trial=1,
        directory=output_subdir,
        project_name='tuning',
        overwrite=True
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

    print("  Buscando mejores hiperparámetros...")
    tuner.search(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=HP_EPOCHS,
        batch_size=32,  # batch fijo durante búsqueda (luego se puede ajustar)
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_params = {
        'units': best_hp.get('units'),
        'dropout': best_hp.get('dropout'),
        'lr': best_hp.get('lr')
    }
    print(f"  Mejores hiperparámetros: {best_params}")

    # 2. Entrenar modelo final con todos los datos (train+val) usando mejores hp
    #    y batch size óptimo (probamos el que mejor fue en tuning, pero como no varió batch, lo fijamos)
    model = build_model(best_params, input_shape)
    # También podríamos hacer una segunda búsqueda de batch size, pero simplificamos

    # Combinar train y validation para entrenamiento final
    X_train_full = np.concatenate([X_train, X_val], axis=0)
    y_train_full = np.concatenate([y_train, y_val], axis=0)

    # Entrenar con early stopping en validation (aunque ahora no tenemos val separada, usamos split)
    history = model.fit(
        X_train_full, y_train_full,
        validation_split=0.1,  # reservamos 10% del full para early stopping
        epochs=HP_EPOCHS,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)],
        verbose=1
    )

    # 3. Evaluación en test (desescalando)
    y_pred_scaled = model.predict(X_test, verbose=0)
    # Invertir normalización
    y_test_descaled = scaler_y.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    y_pred_descaled = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).reshape(y_pred_scaled.shape)

    test_metrics = compute_metrics(y_test_descaled.ravel(), y_pred_descaled.ravel())
    print(f"  Métricas en test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}, MAPE={test_metrics['mape']:.2f}%, Willmott={test_metrics['willmott']:.3f}")

    # 4. Guardar modelo, resultados y gráficas
    model.save(os.path.join(output_subdir, "model.keras"))
    with open(os.path.join(output_subdir, "history.pkl"), 'wb') as f:
        pickle.dump(history.history, f)

    results = {
        'best_params': best_params,
        'test_metrics': test_metrics,
        'n_train': len(X_train_full),
        'n_test': len(X_test)
    }
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    # Gráficas
    horizons = [23, 47, 71]
    plot_predictions(y_test_descaled, y_pred_descaled, horizons,
                     os.path.join(output_subdir, "test_scatter.png"),
                     f"RNN (GRU) - {station_name} - Test")

    # Guardar predicciones
    np.save(os.path.join(output_subdir, "test_pred.npy"), y_pred_descaled)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test_descaled)

    return model, test_metrics

# ============================================================================
# PROCESAMIENTO POR ESTACIÓN
# ============================================================================

def process_by_station():
    print("\n" + "="*50)
    print("PROCESANDO RNN POR ESTACIÓN")
    print("="*50)

    dl_dir = os.path.join(DATA_DIR, "by_station", "dl")
    if not os.path.exists(dl_dir):
        print(f"No existe la carpeta: {dl_dir}")
        return

    stations = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    if not stations:
        print("No se encontraron estaciones con datos DL.")
        return

    for station in stations:
        station_dir = os.path.join(dl_dir, station)
        # Cargar datos
        X_train = np.load(os.path.join(station_dir, "train_X.npy"))
        y_train = np.load(os.path.join(station_dir, "train_y.npy"))
        X_val = np.load(os.path.join(station_dir, "val_X.npy"))
        y_val = np.load(os.path.join(station_dir, "val_y.npy"))
        X_test = np.load(os.path.join(station_dir, "test_X.npy"))
        y_test = np.load(os.path.join(station_dir, "test_y.npy"))

        # Cargar scaler_y
        with open(os.path.join(station_dir, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)

        out_subdir = os.path.join(OUTPUT_DIR, "by_station", station)
        os.makedirs(out_subdir, exist_ok=True)

        try:
            train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                                   scaler_y, station, out_subdir, is_global=False)
        except Exception as e:
            print(f"Error procesando {station}: {e}")
            continue

# ============================================================================
# PROCESAMIENTO GLOBAL
# ============================================================================

def process_global():
    print("\n" + "="*50)
    print("PROCESANDO RNN GLOBAL")
    print("="*50)

    global_dl = os.path.join(DATA_DIR, "global", "dl")
    if not os.path.exists(global_dl):
        print(f"No existe la carpeta: {global_dl}")
        return

    required = ["train_X.npy", "train_y.npy", "val_X.npy", "val_y.npy", "test_X.npy", "test_y.npy"]
    if not all(os.path.exists(os.path.join(global_dl, f)) for f in required):
        print("Faltan archivos en la carpeta global dl.")
        return

    X_train = np.load(os.path.join(global_dl, "train_X.npy"))
    y_train = np.load(os.path.join(global_dl, "train_y.npy"))
    X_val = np.load(os.path.join(global_dl, "val_X.npy"))
    y_val = np.load(os.path.join(global_dl, "val_y.npy"))
    X_test = np.load(os.path.join(global_dl, "test_X.npy"))
    y_test = np.load(os.path.join(global_dl, "test_y.npy"))

    with open(os.path.join(global_dl, "scaler_y.pkl"), 'rb') as f:
        scaler_y = pickle.load(f)

    out_subdir = os.path.join(OUTPUT_DIR, "global")
    os.makedirs(out_subdir, exist_ok=True)

    train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                           scaler_y, "GLOBAL", out_subdir, is_global=True)

# ============================================================================
# RESUMEN FINAL
# ============================================================================

def generate_summary():
    summary = []
    # Por estación
    by_station_dir = os.path.join(OUTPUT_DIR, "by_station")
    if os.path.exists(by_station_dir):
        for station in os.listdir(by_station_dir):
            res_file = os.path.join(by_station_dir, station, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({
                    'station': station,
                    'type': 'by_station',
                    **data['test_metrics']
                })
    # Global
    global_res = os.path.join(OUTPUT_DIR, "global", "results.json")
    if os.path.exists(global_res):
        with open(global_res, 'r') as f:
            data = json.load(f)
        summary.append({
            'station': 'GLOBAL',
            'type': 'global',
            **data['test_metrics']
        })

    if summary:
        df = pd.DataFrame(summary)
        df.to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado en:", os.path.join(OUTPUT_DIR, "summary_metrics.csv"))

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("RNN (ENCODER-DECODER GRU) - ENTRENAMIENTO Y EVALUACIÓN")
    print("="*60)

    process_by_station()
    process_global()
    generate_summary()

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")

# Modelo RNN con metodo por Transecto y metodo General

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 9 (adaptado a transectos): Modelo RNN (encoder-decoder GRU).
Entrada: datasets 3D normalizados (DL) de windows_partitioned/ (por transecto y global).
Salida: modelos entrenados, métricas globales y por estación (para transectos),
        gráficas y archivos de resultados.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, RepeatVector, TimeDistributed, Reshape, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import keras_tuner as kt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
DATA_DIR = os.path.join(BASE_DIR, "windows_partitioned")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")
OUTPUT_DIR = os.path.join(BASE_DIR, "models", "rnn")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parámetros
WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)

# Hiperparámetros a explorar (ajustar según necesidad)
HP_UNITS = [32, 64, 128]
HP_DROPOUT = [0.0, 0.2, 0.3]
HP_LR = [1e-3, 5e-4, 1e-4]
HP_EPOCHS = 100  # máximo, con early stopping
BATCH_SIZE = 32  # fijo durante búsqueda (luego se podría optimizar aparte)

# ============================================================================
# MÉTRICAS
# ============================================================================

def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan

def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def build_model(hp, input_shape, output_dim=72):
    """Construye modelo encoder-decoder GRU con hiperparámetros."""
    if isinstance(hp, dict):
        units = hp['units']
        dropout = hp['dropout']
        lr = hp['lr']
    else:
        units = hp.Choice('units', HP_UNITS)
        dropout = hp.Choice('dropout', HP_DROPOUT)
        lr = hp.Choice('lr', HP_LR)

    # Encoder
    encoder_inputs = Input(shape=input_shape, name='encoder_input')
    encoder = GRU(units, return_state=True, dropout=dropout,
                  recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    encoder_outputs, state_h = encoder(encoder_inputs)

    # Decoder: repetir estado para 72 pasos
    decoder_repeated = RepeatVector(output_dim)(state_h)  # (batch, 72, units)
    decoder_gru = GRU(units, return_sequences=True, dropout=dropout,
                      recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    decoder_outputs = decoder_gru(decoder_repeated, initial_state=state_h)

    # Capa densa por paso
    decoder_dense = TimeDistributed(Dense(1))(decoder_outputs)  # (batch, 72, 1)
    outputs = Reshape((output_dim,))(decoder_dense)  # (batch, 72)

    model = Model(encoder_inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

def get_station_idx_and_mapping(entity_name, transect=True):
    """
    Para una entidad (transecto o estación), obtiene:
    - idx_estacion: índice de la columna 'Estacion' en el DataFrame original (DL).
    - mapping_estacion: dict {código entero: nombre de estación} (si es transecto, necesario para desglose).
    Si es global (transect=False), mapping_estacion es None.
    """
    if transect:
        # Ruta al archivo CSV original (DL) del transecto
        csv_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}.csv")
        mapping_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}_mapping.json")
    else:
        # Para global, no necesitamos mapping (cada entidad es una estación)
        return None, None

    if not os.path.exists(csv_path):
        print(f"    Advertencia: no se encontró {csv_path}")
        return None, None

    # Leer solo los nombres de las columnas
    df_cols = pd.read_csv(csv_path, nrows=0, index_col=0)
    feature_names = df_cols.columns.tolist()
    if 'Estacion' not in feature_names:
        print(f"    Error: no se encuentra columna 'Estacion' en {csv_path}")
        return None, None
    idx_estacion = feature_names.index('Estacion')

    # Cargar mapping
    if os.path.exists(mapping_path):
        with open(mapping_path, 'r') as f:
            mapping_data = json.load(f)
        # mapping_data tiene la forma: {"Estacion": {"nombre_estacion": codigo, ...}}
        # Necesitamos un diccionario código -> nombre
        est_map = mapping_data.get('Estacion', {})
        # Invertir: código (valor) -> nombre (clave)
        mapping_estacion = {int(v): k for k, v in est_map.items()}
    else:
        print(f"    Advertencia: no se encontró mapping en {mapping_path}")
        mapping_estacion = None

    return idx_estacion, mapping_estacion

def compute_per_station_metrics_rnn(y_true, y_pred, X_test, idx_estacion, mapping_estacion, output_dir, entity_name):
    """
    Calcula métricas por estación para un conjunto de test de un transecto.
    X_test: (n_test, window_in, n_features) - con la columna 'Estacion' en idx_estacion.
    mapping_estacion: dict {código entero: nombre estación}
    """
    station_preds = {}
    n_samples = len(y_true)
    for i in range(n_samples):
        # Tomamos el código de estación en el primer paso de la ventana
        station_code = int(round(X_test[i, 0, idx_estacion]))  # puede tener decimales por normalización? pero es entero
        station_name = mapping_estacion.get(station_code, f"Unknown_{station_code}")
        if station_name not in station_preds:
            station_preds[station_name] = {'true': [], 'pred': []}
        station_preds[station_name]['true'].append(y_true[i])
        station_preds[station_name]['pred'].append(y_pred[i])

    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])   # (n_station, 72)
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics

    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
        print(f"    Métricas por estación guardadas en {entity_name}_per_station_metrics.csv")
    return station_metrics

def train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                           scaler_y, entity_name, output_subdir,
                           idx_estacion=None, mapping_estacion=None):
    """
    Entrena RNN con búsqueda de hiperparámetros.
    Si idx_estacion y mapping_estacion se proporcionan, calcula métricas por estación.
    """
    print(f"\n--- Entrenando RNN para {entity_name} ---")
    input_shape = (X_train.shape[1], X_train.shape[2])

    # 1. Búsqueda de hiperparámetros con keras_tuner
    tuner = kt.RandomSearch(
        hypermodel=lambda hp: build_model(hp, input_shape),
        objective='val_loss',
        max_trials=len(HP_UNITS) * len(HP_DROPOUT) * len(HP_LR),
        executions_per_trial=1,
        directory=output_subdir,
        project_name='tuning',
        overwrite=True
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

    print("  Buscando mejores hiperparámetros...")
    tuner.search(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=HP_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_params = {
        'units': best_hp.get('units'),
        'dropout': best_hp.get('dropout'),
        'lr': best_hp.get('lr')
    }
    print(f"  Mejores hiperparámetros: {best_params}")

    # 2. Entrenar modelo final con train+val
    model = build_model(best_params, input_shape)
    X_train_full = np.concatenate([X_train, X_val], axis=0)
    y_train_full = np.concatenate([y_train, y_val], axis=0)

    history = model.fit(
        X_train_full, y_train_full,
        validation_split=0.1,
        epochs=HP_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)],
        verbose=1
    )

    # 3. Evaluación en test (desescalando)
    y_pred_scaled = model.predict(X_test, verbose=0)
    y_test_descaled = scaler_y.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    y_pred_descaled = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).reshape(y_pred_scaled.shape)

    test_metrics = compute_metrics(y_test_descaled.ravel(), y_pred_descaled.ravel())
    print(f"  Métricas en test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, "
          f"RMSE={test_metrics['rmse']:.2f}, MAPE={test_metrics['mape']:.2f}%, Willmott={test_metrics['willmott']:.3f}")

    # 4. Métricas por estación (si aplica)
    station_metrics = None
    if idx_estacion is not None and mapping_estacion is not None:
        station_metrics = compute_per_station_metrics_rnn(
            y_test_descaled, y_pred_descaled, X_test, idx_estacion, mapping_estacion,
            output_subdir, entity_name
        )

    # 5. Guardar modelo y resultados
    model.save(os.path.join(output_subdir, "model.keras"))
    with open(os.path.join(output_subdir, "history.pkl"), 'wb') as f:
        pickle.dump(history.history, f)

    results = {
        'best_params': best_params,
        'test_metrics': test_metrics,
        'station_metrics': station_metrics,
        'n_train': len(X_train_full),
        'n_test': len(X_test)
    }
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    # 6. Gráficas
    horizons = [23, 47, 71]
    plot_predictions(y_test_descaled, y_pred_descaled, horizons,
                     os.path.join(output_subdir, "test_scatter.png"),
                     f"RNN (GRU) - {entity_name} - Test")

    np.save(os.path.join(output_subdir, "test_pred.npy"), y_pred_descaled)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test_descaled)

    return model, test_metrics

# ============================================================================
# PROCESAMIENTO POR TRANSECTO
# ============================================================================

def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO RNN POR TRANSECTO")
    print("="*50)

    dl_dir = os.path.join(DATA_DIR, "by_transect", "dl")
    if not os.path.exists(dl_dir):
        print(f"No existe la carpeta: {dl_dir}")
        return

    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    if not entities:
        print("No se encontraron transectos.")
        return

    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        # Cargar datos
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)

        # Obtener índice de 'Estacion' y mapping
        idx_estacion, mapping_estacion = get_station_idx_and_mapping(entity, transect=True)
        if idx_estacion is None:
            print(f"  No se pudo determinar la columna Estacion para {entity}. Se omitirá el desglose por estación.")

        # Crear subcarpeta de salida
        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)

        try:
            train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                                   scaler_y, entity, out_subdir,
                                   idx_estacion, mapping_estacion)
        except Exception as e:
            print(f"Error procesando {entity}: {e}")
            continue

# ============================================================================
# PROCESAMIENTO GLOBAL (por estación)
# ============================================================================

def process_global():
    print("\n" + "="*50)
    print("PROCESANDO RNN GLOBAL (POR ESTACIÓN)")
    print("="*50)

    dl_dir = os.path.join(DATA_DIR, "global", "dl")
    if not os.path.exists(dl_dir):
        print(f"No existe la carpeta: {dl_dir}")
        return

    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    if not entities:
        print("No se encontraron estaciones.")
        return

    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)

        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)

        try:
            train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                                   scaler_y, entity, out_subdir,
                                   idx_estacion=None, mapping_estacion=None)
        except Exception as e:
            print(f"Error procesando {entity}: {e}")
            continue

# ============================================================================
# RESUMEN FINAL
# ============================================================================

def generate_summary():
    summary = []
    # Por transecto
    transect_dir = os.path.join(OUTPUT_DIR, "by_transect")
    if os.path.exists(transect_dir):
        for entity in os.listdir(transect_dir):
            res_file = os.path.join(transect_dir, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({
                    'entity': entity,
                    'type': 'by_transect',
                    **data['test_metrics']
                })
    # Global
    global_dir = os.path.join(OUTPUT_DIR, "global")
    if os.path.exists(global_dir):
        for entity in os.listdir(global_dir):
            res_file = os.path.join(global_dir, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({
                    'entity': entity,
                    'type': 'global',
                    **data['test_metrics']
                })
    if summary:
        df = pd.DataFrame(summary)
        df.to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado en:", os.path.join(OUTPUT_DIR, "summary_metrics.csv"))

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("RNN (ENCODER-DECODER GRU) - ENTRENAMIENTO Y EVALUACIÓN (TRANSECTOS + GLOBAL)")
    print("="*60)

    process_by_transect()
    process_global()
    generate_summary()

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")

Trial 2 Complete [00h 03m 09s]
val_loss: nan

Best val_loss So Far: nan
Total elapsed time: 00h 07m 06s

Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
128               |64                |units
0                 |0.2               |dropout
0.0005            |0.0001            |lr

Epoch 1/100
408/408 ━━━━━━━━━━━━━━━━━━━━ 37s 81ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 5.0000e-04
Epoch 2/100
408/408 ━━━━━━━━━━━━━━━━━━━━ 32s 79ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 5.0000e-04
Epoch 3/100
408/408 ━━━━━━━━━━━━━━━━━━━━ 32s 78ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 5.0000e-04
Epoch 4/100
408/408 ━━━━━━━━━━━━━━━━━━━━ 33s 80ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 5.0000e-04
Epoch 5/100
408/408 ━━━━━━━━━━━━━━━━━━━━ 32s 79ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 5.0000e-04